# Retriever training

In [ ]:
# Colab/local environment setup
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Help CUDA allocator reduce fragmentation (must be set before importing torch).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Keep this False for normal training speed. Turn on only when debugging CUDA asserts.
DEBUG_CUDA_ASSERT = False
if DEBUG_CUDA_ASSERT:
    os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

if IN_COLAB:
    print("Running in Google Colab")
    # Install missing dependencies quietly when needed.
    !pip -q install transformers scikit-learn tqdm peft
    
    # Optional: mount Drive so data/artifacts can persist.
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Running in local environment")

print(f"PYTORCH_CUDA_ALLOC_CONF={os.environ.get('PYTORCH_CUDA_ALLOC_CONF')}")

if DEBUG_CUDA_ASSERT:
    print("CUDA_LAUNCH_BLOCKING=1 enabled for debugging.")
    print("After fixing errors, set DEBUG_CUDA_ASSERT=False for faster training.")
else:
    print("CUDA_LAUNCH_BLOCKING disabled for faster training.")

In [ ]:
import json
from pathlib import Path


def get_jsonl_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data


def resolve_retriever_data_path() -> str:
    # 1) Prefer repository-relative path (local VS Code, Colab after git clone).
    candidates = [
        Path("data/retriever_data.jsonl"),
        Path("/content/drive/MyDrive/Colab Notebooks/NLP/Legal-question-answer-v1/data/retriever_data.jsonl"),
    ]

    for p in candidates:
        if p.exists():
            return str(p)

    raise FileNotFoundError(
        "Cannot find retriever_data.jsonl. Place it under data/ or update the candidate paths."
    )


retriever_data_jsonl_path = resolve_retriever_data_path()
print(f"Loading retriever data from: {retriever_data_jsonl_path}")
retriever_data = get_jsonl_data(retriever_data_jsonl_path)
retriever_data[0]

In [ ]:
import random
import copy
import math
from dataclasses import dataclass, field
from typing import Dict, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, TaskType, get_peft_model

# ----------------------------
# 1) Reproducibility settings
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    # Slight speed/memory win on Ampere+ without changing model outputs materially.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

@dataclass
class TrainConfig:
    model_name: str = "vinai/phobert-base"
    # Use shorter lengths for faster iteration. Increase later if recall drops.
    max_q_len: int = 96
    max_c_len: int = 128
    batch_size: int = 16
    grad_accum_steps: int = 2
    max_epochs: int = 12
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    temperature: float = 0.05
    grad_clip: float = 1.0
    warmup_ratio: float = 0.1
    early_stopping_patience: int = 2
    num_negatives: int = 1
    refresh_every_epochs: int = 2
    dataloader_num_workers: int = 8
    dataloader_persistent_workers: bool = True
    # LoRA hyperparameters for PhoBERT (RoBERTa-style attention blocks).
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.1
    lora_target_modules: List[str] = field(default_factory=lambda: ["query", "key", "value"])
    debug_checks: bool = False
    log_every: int = 20


config = TrainConfig()
if config.lora_r <= 0:
    raise ValueError("lora_r must be > 0")
if config.lora_alpha <= 0:
    raise ValueError("lora_alpha must be > 0")

print(config)
print(f"Effective train batch size = {config.batch_size * config.grad_accum_steps}")
print(
    f"LoRA setup | r={config.lora_r}, alpha={config.lora_alpha}, "
    f"dropout={config.lora_dropout}, targets={config.lora_target_modules}"
)
print(
    f"DataLoader workers={config.dataloader_num_workers}, "
    f"persistent_workers={config.dataloader_persistent_workers}"
)

In [ ]:
# ---------------------------------------------------------------------
# 2) Convert raw records to training pairs: (question, positive, negatives)
#    Current data schema: question, positive_chunks, negative_chunks
# ---------------------------------------------------------------------
def build_training_pairs(data: List[Dict]) -> List[Dict]:
    pairs = []
    for row in data:
        question = row.get("question", "")
        positives = row.get("positive_chunks", [])
        negatives = row.get("negative_chunks", [])

        if not question or not positives:
            continue

        for pos in positives:
            if isinstance(pos, str) and pos.strip():
                pairs.append(
                    {
                        "question": question,
                        "positive": pos,
                        "negatives": [n for n in negatives if isinstance(n, str) and n.strip()],
                    }
                )
    return pairs


all_pairs = build_training_pairs(retriever_data)
print(f"Total (question, positive) pairs: {len(all_pairs)}")

# --------------------------------------------------------
# 3) Split data train/val/test with ratio 70/15/15
# --------------------------------------------------------
train_pairs, temp_pairs = train_test_split(
    all_pairs,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
    stratify=None,
    )

val_pairs, test_pairs = train_test_split(
    temp_pairs,
    test_size=0.50,
    random_state=SEED,
    shuffle=True,
    stratify=None,
    )

print(f"Train size: {len(train_pairs)} ({len(train_pairs)/len(all_pairs):.2%})")
print(f"Val size:   {len(val_pairs)} ({len(val_pairs)/len(all_pairs):.2%})")
print(f"Test size:  {len(test_pairs)} ({len(test_pairs)/len(all_pairs):.2%})")

In [ ]:
# --------------------------------------------------------------------
# 4) Dataset with negatives refresh rule (with pre-tokenization)
#    - use only config.num_negatives negatives each time
#    - refresh every config.refresh_every_epochs epochs
#    - if not enough negatives for next refresh, keep old negatives
# --------------------------------------------------------------------
class RetrieverPairDataset(Dataset):
    def __init__(
        self,
        pairs: List[Dict],
        tokenizer,
        max_q_len: int,
        max_c_len: int,
        num_negatives: int = 2,
    ):
        self.pairs = pairs
        self.num_negatives = num_negatives
        self.refresh_cycle = 0

        # Batched pre-tokenization
        questions = [row["question"] for row in self.pairs]
        positives = [row["positive"] for row in self.pairs]

        q_batch = tokenizer(
            questions,
            padding="max_length",
            truncation=True,
            max_length=max_q_len,
            return_tensors="pt",
        )
        p_batch = tokenizer(
            positives,
            padding="max_length",
            truncation=True,
            max_length=max_c_len,
            return_tensors="pt",
        )

        negative_texts = []
        negative_spans = []
        for row in self.pairs:
            start = len(negative_texts)
            row_negs = row["negatives"]
            negative_texts.extend(row_negs)
            end = len(negative_texts)
            negative_spans.append((start, end))

        if negative_texts:
            n_batch = tokenizer(
                negative_texts,
                padding="max_length",
                truncation=True,
                max_length=max_c_len,
                return_tensors="pt",
            )
        else:
            n_batch = None

        self.encoded_pairs = []
        for i, row in enumerate(self.pairs):
            neg_start, neg_end = negative_spans[i]
            encoded_negatives = []

            if n_batch is not None:
                for n_idx in range(neg_start, neg_end):
                    encoded_negatives.append(
                        {
                            "text": negative_texts[n_idx],
                            "input_ids": n_batch["input_ids"][n_idx],
                            "attention_mask": n_batch["attention_mask"][n_idx],
                        }
                    )

            self.encoded_pairs.append(
                {
                    "question": row["question"],
                    "positive": row["positive"],
                    "q_tok": {
                        "input_ids": q_batch["input_ids"][i],
                        "attention_mask": q_batch["attention_mask"][i],
                    },
                    "p_tok": {
                        "input_ids": p_batch["input_ids"][i],
                        "attention_mask": p_batch["attention_mask"][i],
                    },
                    "encoded_negatives": encoded_negatives,
                }
            )

    def set_refresh_cycle(self, cycle: int):
        self.refresh_cycle = cycle

    def _select_negatives(self, negatives: List[Dict], cycle: int) -> List[Dict]:
        if not negatives:
            return []

        window = self.num_negatives
        start = cycle * window
        end = start + window

        if end <= len(negatives):
            return negatives[start:end]

        if cycle > 0:
            prev_start = (cycle - 1) * window
            prev_end = prev_start + window
            if prev_end <= len(negatives):
                return negatives[prev_start:prev_end]

        return negatives[: min(window, len(negatives))]

    def __len__(self):
        return len(self.encoded_pairs)

    def __getitem__(self, idx):
        row = self.encoded_pairs[idx]
        selected_negatives = self._select_negatives(row["encoded_negatives"], self.refresh_cycle)

        return {
            "question": row["question"],
            "positive": row["positive"],
            "negatives": [n["text"] for n in selected_negatives],
            "q_tok": row["q_tok"],
            "p_tok": row["p_tok"],
            "n_tok_list": [
                {
                    "input_ids": n["input_ids"],
                    "attention_mask": n["attention_mask"],
                }
                for n in selected_negatives
            ],
        }


tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

# RoBERTa/PhoBERT positional embedding size is typically 258, meaning safe max tokenized length is 256.
model_max_len = getattr(tokenizer, "model_max_length", None)
if isinstance(model_max_len, int) and model_max_len > 0 and model_max_len < 10_000:
    if config.max_q_len > model_max_len or config.max_c_len > model_max_len:
        raise ValueError(
            f"max_q_len/max_c_len must be <= tokenizer.model_max_length ({model_max_len}). "
            f"Got max_q_len={config.max_q_len}, max_c_len={config.max_c_len}."
        )
    print(f"Tokenizer max length: {model_max_len} | q_len={config.max_q_len}, c_len={config.max_c_len}")
else:
    # Conservative fallback for PhoBERT-like models if tokenizer does not expose a practical value.
    if config.max_q_len > 256 or config.max_c_len > 256:
        raise ValueError(
            f"For PhoBERT, set max_q_len/max_c_len <= 256. "
            f"Got max_q_len={config.max_q_len}, max_c_len={config.max_c_len}."
        )


def collate_fn(batch: List[Dict]):
    q_tok = {
        "input_ids": torch.stack([x["q_tok"]["input_ids"] for x in batch], dim=0),
        "attention_mask": torch.stack([x["q_tok"]["attention_mask"] for x in batch], dim=0),
    }
    p_tok = {
        "input_ids": torch.stack([x["p_tok"]["input_ids"] for x in batch], dim=0),
        "attention_mask": torch.stack([x["p_tok"]["attention_mask"] for x in batch], dim=0),
    }

    flat_negatives = [n for x in batch for n in x["n_tok_list"]]
    neg_counts = [len(x["n_tok_list"]) for x in batch]

    if len(flat_negatives) > 0:
        n_tok = {
            "input_ids": torch.stack([n["input_ids"] for n in flat_negatives], dim=0),
            "attention_mask": torch.stack([n["attention_mask"] for n in flat_negatives], dim=0),
        }
    else:
        n_tok = None

    return {
        "q_tok": q_tok,
        "p_tok": p_tok,
        "n_tok": n_tok,
        "neg_counts": neg_counts,
    }

In [ ]:
import time
from contextlib import nullcontext
from tqdm.auto import tqdm


class DualEncoder(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        # Dual-encoder: one encoder for questions, one encoder for chunks.
        self.q_encoder = AutoModel.from_pretrained(model_name)
        self.d_encoder = AutoModel.from_pretrained(model_name)

    @staticmethod
    def mean_pool(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = torch.sum(last_hidden_state * mask, dim=1)
        denom = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / denom

    @staticmethod
    def _validate_token_ids(tok, vocab_size: int, name: str):
        input_ids = tok["input_ids"]
        min_id = int(input_ids.min().item())
        max_id = int(input_ids.max().item())
        if min_id < 0 or max_id >= vocab_size:
            raise RuntimeError(
                f"{name} has out-of-range token ids: min={min_id}, max={max_id}, vocab_size={vocab_size}"
            )

    def encode_questions(self, q_tok):
        out = self.q_encoder(
            input_ids=q_tok["input_ids"],
            attention_mask=q_tok["attention_mask"],
        )
        emb = self.mean_pool(out.last_hidden_state, q_tok["attention_mask"])
        return F.normalize(emb, p=2, dim=-1)

    def encode_docs(self, d_tok):
        out = self.d_encoder(
            input_ids=d_tok["input_ids"],
            attention_mask=d_tok["attention_mask"],
        )
        emb = self.mean_pool(out.last_hidden_state, d_tok["attention_mask"])
        return F.normalize(emb, p=2, dim=-1)

    def forward(self, batch):
        q_tok = {k: v.to(device) for k, v in batch["q_tok"].items()}
        p_tok = {k: v.to(device) for k, v in batch["p_tok"].items()}
        n_tok = batch["n_tok"]
        neg_counts = batch["neg_counts"]

        if config.debug_checks:
            q_vocab_size = self.q_encoder.config.vocab_size
            d_vocab_size = self.d_encoder.config.vocab_size
            self._validate_token_ids(q_tok, q_vocab_size, "Question tokens")
            self._validate_token_ids(p_tok, d_vocab_size, "Positive doc tokens")

        q_emb = self.encode_questions(q_tok)
        p_emb = self.encode_docs(p_tok)

        if n_tok is not None:
            n_tok = {k: v.to(device) for k, v in n_tok.items()}
            if config.debug_checks:
                d_vocab_size = self.d_encoder.config.vocab_size
                self._validate_token_ids(n_tok, d_vocab_size, "Negative doc tokens")
            n_emb = self.encode_docs(n_tok)
        else:
            n_emb = None

        losses = []
        correct = 0
        start = 0

        for i in range(q_emb.size(0)):
            candidates = [p_emb[i].unsqueeze(0)]
            n_count = neg_counts[i]

            if n_count > 0 and n_emb is not None:
                cur_negs = n_emb[start : start + n_count]
                candidates.append(cur_negs)
                start += n_count

            c_emb = torch.cat(candidates, dim=0)
            logits = torch.matmul(q_emb[i].unsqueeze(0), c_emb.transpose(0, 1)) / config.temperature
            labels = torch.zeros(1, dtype=torch.long, device=device)

            if config.debug_checks and not torch.isfinite(logits).all():
                raise RuntimeError("Non-finite logits detected (NaN/Inf).")

            loss_i = F.cross_entropy(logits, labels)
            if config.debug_checks and not torch.isfinite(loss_i):
                raise RuntimeError("Non-finite loss detected (NaN/Inf).")

            losses.append(loss_i)

            pred = torch.argmax(logits, dim=1).item()
            if pred == 0:
                correct += 1

        loss = torch.stack(losses).mean()
        acc = correct / q_emb.size(0)
        return loss, acc


def run_epoch(
    model,
    loader,
    optimizer=None,
    scheduler=None,
    train_mode=True,
    scaler=None,
    grad_accum_steps: int = 1,
    use_amp: bool = True,
    amp_dtype: torch.dtype = torch.float16,
    epoch_idx: int = 0,
    show_progress: bool = True,
    log_every: int = 20,
):
    if train_mode:
        model.train()
        if optimizer is None:
            raise ValueError("optimizer is required when train_mode=True")
        optimizer.zero_grad(set_to_none=True)
    else:
        model.eval()

    total_loss = 0.0
    total_acc = 0.0
    steps = 0

    autocast_enabled = bool(use_amp and torch.cuda.is_available())
    phase = "train" if train_mode else "val/test"
    pbar = tqdm(
        enumerate(loader),
        total=len(loader),
        desc=f"Epoch {epoch_idx:02d} [{phase}]",
        leave=False,
        disable=not show_progress,
    )

    for step_idx, batch in pbar:
        try:
            with torch.set_grad_enabled(train_mode):
                autocast_ctx = (
                    torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=True)
                    if autocast_enabled
                    else nullcontext()
                )
                with autocast_ctx:
                    loss, acc = model(batch)

                if train_mode:
                    loss_to_backprop = loss / max(grad_accum_steps, 1)

                    if scaler is not None and autocast_enabled:
                        scaler.scale(loss_to_backprop).backward()
                    else:
                        loss_to_backprop.backward()

                    should_step = (
                        ((step_idx + 1) % max(grad_accum_steps, 1) == 0)
                        or ((step_idx + 1) == len(loader))
                    )

                    if should_step:
                        if scaler is not None and autocast_enabled:
                            old_scale = scaler.get_scale()
                            scaler.unscale_(optimizer)
                            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
                            scaler.step(optimizer)
                            scaler.update()
                            optimizer_was_run = scaler.get_scale() >= old_scale
                        else:
                            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
                            optimizer.step()
                            optimizer_was_run = True

                        optimizer.zero_grad(set_to_none=True)
                        if scheduler is not None and optimizer_was_run:
                            scheduler.step()

            total_loss += loss.item()
            total_acc += acc
            steps += 1

            if steps % max(log_every, 1) == 0 or steps == len(loader):
                avg_loss = total_loss / max(steps, 1)
                avg_acc = total_acc / max(steps, 1)
                pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{avg_acc:.4f}")

        except RuntimeError as e:
            msg = str(e)
            if "device-side assert" in msg or "out-of-range token ids" in msg or "Non-finite" in msg:
                print(f"Runtime error at step {step_idx}: {msg}")
                print("Hint: inspect raw batch texts and token ranges for this step.")
            raise

    return total_loss / max(steps, 1), total_acc / max(steps, 1)

In [ ]:
# -------------------------
# 5) Build dataloaders
# -------------------------
train_ds = RetrieverPairDataset(
    train_pairs,
    tokenizer=tokenizer,
    max_q_len=config.max_q_len,
    max_c_len=config.max_c_len,
    num_negatives=config.num_negatives,
 )
val_ds = RetrieverPairDataset(
    val_pairs,
    tokenizer=tokenizer,
    max_q_len=config.max_q_len,
    max_c_len=config.max_c_len,
    num_negatives=config.num_negatives,
 )
test_ds = RetrieverPairDataset(
    test_pairs,
    tokenizer=tokenizer,
    max_q_len=config.max_q_len,
    max_c_len=config.max_c_len,
    num_negatives=config.num_negatives,
 )

num_workers = max(0, int(config.dataloader_num_workers))
use_persistent_workers = bool(config.dataloader_persistent_workers and num_workers > 0)

train_loader = DataLoader(
    train_ds,
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
    num_workers=num_workers,
    persistent_workers=use_persistent_workers,
)
val_loader = DataLoader(
    val_ds,
    batch_size=config.batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
    num_workers=num_workers,
    persistent_workers=use_persistent_workers,
)
test_loader = DataLoader(
    test_ds,
    batch_size=config.batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
    num_workers=num_workers,
    persistent_workers=use_persistent_workers,
)

# -------------------------------
# 6) Model + LoRA + optimizers
# -------------------------------
model = DualEncoder(config.model_name).to(device)

if hasattr(model.q_encoder, "gradient_checkpointing_enable"):
    model.q_encoder.gradient_checkpointing_enable()
if hasattr(model.d_encoder, "gradient_checkpointing_enable"):
    model.d_encoder.gradient_checkpointing_enable()

# Transformers warning fix: use_cache must be disabled with gradient checkpointing.
if hasattr(model.q_encoder, "config") and hasattr(model.q_encoder.config, "use_cache"):
    model.q_encoder.config.use_cache = False
if hasattr(model.d_encoder, "config") and hasattr(model.d_encoder.config, "use_cache"):
    model.d_encoder.config.use_cache = False


def apply_lora_to_encoder(hf_model, cfg: TrainConfig):
    lora_cfg = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=cfg.lora_target_modules,
        bias="none",
    )
    return get_peft_model(hf_model, lora_cfg)


model.q_encoder = apply_lora_to_encoder(model.q_encoder, config)
model.d_encoder = apply_lora_to_encoder(model.d_encoder, config)

# Keep only LoRA adapter params trainable (PEFT handles this, but assert for safety).
trainable_params = [p for p in model.parameters() if p.requires_grad]
if not trainable_params:
    raise RuntimeError("No trainable parameters found after applying LoRA.")

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)

steps_per_epoch = math.ceil(len(train_loader) / config.grad_accum_steps)
total_training_steps = steps_per_epoch * config.max_epochs
warmup_steps = int(total_training_steps * config.warmup_ratio)
linear_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1,
    min_lr=1e-7,
)

# AMP policy: prefer BF16 on supported GPUs; fallback to FP16 + GradScaler on older CUDA.
if torch.cuda.is_available():
    if torch.cuda.is_bf16_supported():
        use_amp = True
        amp_dtype = torch.bfloat16
        scaler = None  # BF16 does not need gradient scaling.
        amp_mode = "bf16"
    else:
        use_amp = True
        amp_dtype = torch.float16
        scaler = torch.amp.GradScaler("cuda", enabled=True)
        amp_mode = "fp16"
else:
    use_amp = False
    amp_dtype = torch.float32
    scaler = None
    amp_mode = "off (cpu)"

total_params = sum(p.numel() for p in model.parameters())
trainable_param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable_ratio = 100.0 * trainable_param_count / max(total_params, 1)

print(f"Total optimizer update steps: {total_training_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"AMP mode: {amp_mode}")
print(f"Gradient accumulation steps: {config.grad_accum_steps}")
print(f"LoRA target modules: {config.lora_target_modules}")
print(f"Trainable params: {trainable_param_count:,} / {total_params:,} ({trainable_ratio:.2f}%)")
print(
    f"use_cache flags | q_encoder={getattr(model.q_encoder.config, 'use_cache', None)} | "
    f"d_encoder={getattr(model.d_encoder.config, 'use_cache', None)}"
)

# Optional sanity summary from PEFT wrappers.
try:
    model.q_encoder.print_trainable_parameters()
    model.d_encoder.print_trainable_parameters()
except Exception:
    pass

In [ ]:
# -------------------------------
# 7) Train + validation + test
# -------------------------------
best_val_loss = float("inf")
best_state = None
epochs_no_improve = 0
neg_refresh_cycle = 0
history = []

for epoch in range(1, config.max_epochs + 1):
    epoch_start = time.time()

    if epoch > 1 and (epoch - 1) % config.refresh_every_epochs == 0:
        neg_refresh_cycle += 1

    train_ds.set_refresh_cycle(neg_refresh_cycle)
    val_ds.set_refresh_cycle(neg_refresh_cycle)
    test_ds.set_refresh_cycle(neg_refresh_cycle)

    train_loss, train_acc = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scheduler=linear_scheduler,
        train_mode=True,
        scaler=scaler,
        grad_accum_steps=config.grad_accum_steps,
        use_amp=use_amp,
        amp_dtype=amp_dtype,
        epoch_idx=epoch,
        show_progress=True,
        log_every=config.log_every,
    )

    with torch.no_grad():
        val_loss, val_acc = run_epoch(
            model,
            val_loader,
            train_mode=False,
            use_amp=use_amp,
            amp_dtype=amp_dtype,
            epoch_idx=epoch,
            show_progress=True,
            log_every=config.log_every,
        )

    plateau_scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]
    epoch_sec = time.time() - epoch_start

    history.append(
        {
            "epoch": epoch,
            "neg_refresh_cycle": neg_refresh_cycle,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "lr": current_lr,
            "epoch_seconds": epoch_sec,
        }
    )

    print(
        f"Epoch {epoch:02d} | cycle={neg_refresh_cycle} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"lr={current_lr:.2e} | time={epoch_sec/60:.2f} min"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= config.early_stopping_patience:
            print(f"Early stopping triggered at epoch {epoch}.")
            break

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if best_state is not None:
    model.load_state_dict(best_state)

with torch.no_grad():
    test_loss, test_acc = run_epoch(
        model,
        test_loader,
        train_mode=False,
        use_amp=use_amp,
        amp_dtype=amp_dtype,
        epoch_idx=epoch,
        show_progress=True,
        log_every=config.log_every,
    )

print("\nFinal evaluation on test set:")
print(f"Test loss: {test_loss:.4f}")
print(f"Test acc:  {test_acc:.4f}")

In [ ]:
history

In [ ]:
# Save and download both LoRA adapters (Colab + local friendly)
import os
import shutil
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Persist artifacts on Google Drive if mounted.
    drive_root = "/content/drive/MyDrive"
    save_root = os.path.join(drive_root, "artifacts_dual_encoder_lora")
else:
    save_root = "artifacts_dual_encoder_lora"

q_dir = os.path.join(save_root, "question_encoder_lora")
d_dir = os.path.join(save_root, "document_encoder_lora")

if "model" not in globals() or "tokenizer" not in globals():
    print("Run the training/evaluation cells first so trained model/tokenizer are available.")
else:
    os.makedirs(save_root, exist_ok=True)

    # For PEFT models, save_pretrained stores LoRA adapter weights + adapter config.
    model.q_encoder.save_pretrained(q_dir)
    tokenizer.save_pretrained(q_dir)

    model.d_encoder.save_pretrained(d_dir)
    tokenizer.save_pretrained(d_dir)

    q_zip = shutil.make_archive(q_dir, "zip", q_dir)
    d_zip = shutil.make_archive(d_dir, "zip", d_dir)

    print("Saved LoRA adapter folders:")
    print(f"- {q_dir}")
    print(f"- {d_dir}")
    print("Created zip files:")
    print(f"- {q_zip}")
    print(f"- {d_zip}")
    print("Note: these are adapters, so load them on top of the same PhoBERT base model.")

    # If running in Google Colab, trigger browser download automatically.
    if IN_COLAB:
        try:
            from google.colab import files  # type: ignore
            files.download(q_zip)
            files.download(d_zip)
        except Exception:
            print("Colab auto-download unavailable. Use the printed zip paths.")

In [ ]:
# Error analysis: inspect wrong samples on test set
def _short_text(text: str, max_len: int = 220) -> str:
    text = " ".join(text.split())
    if len(text) <= max_len:
        return text
    return text[:max_len] + "..."


def collect_wrong_test_samples(model, test_ds, tokenizer, device, config):
    model.eval()
    wrong_cases = []

    with torch.no_grad():
        for idx in range(len(test_ds)):
            sample = test_ds[idx]
            candidates = [sample["positive"]] + sample["negatives"]

            if len(candidates) == 1:
                # No negatives for this sample -> cannot become a wrong ranking case.
                continue

            q_tok = tokenizer(
                [sample["question"]],
                padding=True,
                truncation=True,
                max_length=config.max_q_len,
                return_tensors="pt",
            )
            c_tok = tokenizer(
                candidates,
                padding=True,
                truncation=True,
                max_length=config.max_c_len,
                return_tensors="pt",
            )

            q_tok = {k: v.to(device) for k, v in q_tok.items()}
            c_tok = {k: v.to(device) for k, v in c_tok.items()}

            q_emb = model.encode_questions(q_tok)  # [1, H]
            c_emb = model.encode_docs(c_tok)       # [num_candidates, H]

            logits = torch.matmul(q_emb, c_emb.transpose(0, 1)).squeeze(0) / config.temperature
            probs = torch.softmax(logits, dim=0)

            pred_idx = int(torch.argmax(logits).item())
            if pred_idx != 0:
                wrong_cases.append(
                    {
                        "test_index": idx,
                        "question": sample["question"],
                        "positive_chunk": sample["positive"],
                        "predicted_index": pred_idx,
                        "predicted_chunk": candidates[pred_idx],
                        "positive_score": float(logits[0].item()),
                        "predicted_score": float(logits[pred_idx].item()),
                        "score_gap": float((logits[pred_idx] - logits[0]).item()),
                        "positive_prob": float(probs[0].item()),
                        "predicted_prob": float(probs[pred_idx].item()),
                        "num_negatives": len(sample["negatives"]),
                    }
                )

    return wrong_cases


if "model" not in globals() or "test_ds" not in globals():
    print("Run the training + test evaluation cells first, then run this cell.")
else:
    wrong_cases = collect_wrong_test_samples(model, test_ds, tokenizer, device, config)
    total_eval = len([test_ds[i] for i in range(len(test_ds)) if len(test_ds[i]["negatives"]) > 0])
    wrong_count = len(wrong_cases)
    wrong_rate = (wrong_count / total_eval * 100) if total_eval > 0 else 0.0

    print(f"Evaluated test samples (with >=1 negative): {total_eval}")
    print(f"Wrong ranking cases: {wrong_count} ({wrong_rate:.2f}%)")

    preview_n = 10
    for i, case in enumerate(wrong_cases[:preview_n], start=1):
        print("\n" + "=" * 90)
        print(f"Case {i} | test_index={case['test_index']} | num_negatives={case['num_negatives']}")
        print(f"Score gap (pred - pos): {case['score_gap']:.4f}")
        print(f"Positive prob: {case['positive_prob']:.4f} | Pred prob: {case['predicted_prob']:.4f}")
        print("Question:", _short_text(case["question"]))
        print("Positive:", _short_text(case["positive_chunk"]))
        print("Predicted:", _short_text(case["predicted_chunk"]))

    wrong_cases